# FinReasoning — standalone evaluation

Run this notebook to evaluate a fine-tuned adapter **with valid metrics**.

**Why Step 4 in the main Colab notebook looked broken:** `data/processed` stores only `prompt`, `completion`, and `task` for SFT. The evaluator needs `answer`, `question`, `context`, and (for numerical tasks) `expression` / `variables`. Loading the test split from disk dropped those fields, so prompts were empty, predictions became *Insufficient information.*, and ground truth appeared empty (NaN in CSV).

**This notebook** rebuilds the **test split from raw JSONL** (same stratified split as preprocessing) and uses the fixed evaluator that builds prompts with `format_as_prompt_completion` (matching training).

**Requirements:** GPU recommended; raw training JSONL under `data/raw/` (or set `RAW_DATA_PATH`); adapter at `outputs/sft_qlora/final_adapter` or set `ADAPTER_DIR`.

**Google Colab:** Run the **Setup** cell below first (mount Drive, clone/pull repo). Paths default to `MyDrive/FinReasoningAI/` like the main training notebook.

## Setup (Google Colab)

Mount Drive and clone or pull **[juankim834/FinReasoningAI](https://github.com/juankim834/FinReasoningAI)** under `MyDrive/FinReasoningAI/FinReasoningAI`. Edit `WORKSPACE` in the next cell if your Drive layout differs.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

REPO_URL = "https://github.com/juankim834/FinReasoningAI.git"
WORKSPACE = "/content/drive/MyDrive/FinReasoningAI"
PROJECT_DIR = os.path.join(WORKSPACE, "FinReasoningAI")

Path(WORKSPACE).mkdir(parents=True, exist_ok=True)

if Path(PROJECT_DIR, ".git").is_dir():
    subprocess.run(["git", "-C", PROJECT_DIR, "pull"], check=False)
else:
    subprocess.run(["git", "clone", REPO_URL, PROJECT_DIR], check=True)

os.chdir(PROJECT_DIR)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)
print("Working directory:", os.getcwd())

Mounted at /content/drive
Working directory: /content/drive/MyDrive/FinReasoningAI/FinReasoningAI


In [ ]:
import os, sys, gc
from pathlib import Path

ROOT = Path.cwd().resolve()
os.chdir(ROOT)
rp = str(ROOT)
if rp not in sys.path:
    sys.path.insert(0, rp)
print("Working directory:", ROOT)

Working directory: /content/drive/MyDrive/FinReasoningAI/FinReasoningAI


## Configuration

Defaults: raw JSONL under the Drive workspace `.../FinReasoningAI/data/raw`, adapter under `.../FinReasoningAI/outputs/sft_qlora/final_adapter` (same as the main Colab symlinks). Set `RAW_DATA_PATH` / `ADAPTER_DIR` environment variables to override.

In [ ]:
MODEL_ID = os.environ.get("MODEL_ID", "Qwen/Qwen2.5-14B-Instruct")

_drive_workspace = ROOT.parent
RAW_DATA_PATH = os.environ.get(
    "RAW_DATA_PATH", str(_drive_workspace / "data" / "raw")
)
ADAPTER_DIR = os.environ.get(
    "ADAPTER_DIR",
    str(_drive_workspace / "outputs" / "sft_qlora" / "final_adapter"),
)

OUTPUT_CSV = "outputs/eval_results.csv"
MAX_SAMPLES = None  # e.g. 50 for a quick smoke test; None = all test samples
MAX_NEW_TOKENS = 128

assert Path(RAW_DATA_PATH).exists(), f"Missing RAW_DATA_PATH: {RAW_DATA_PATH}"
assert Path(ADAPTER_DIR).exists(), f"Missing ADAPTER_DIR: {ADAPTER_DIR}"

print("MODEL_ID:", MODEL_ID)
print("RAW_DATA_PATH:", RAW_DATA_PATH)
print("ADAPTER_DIR:", ADAPTER_DIR)

MODEL_ID: Qwen/Qwen2.5-14B-Instruct
RAW_DATA_PATH: /content/drive/MyDrive/FinReasoningAI/data/raw
ADAPTER_DIR: /content/drive/MyDrive/FinReasoningAI/outputs/sft_qlora/final_adapter


In [ ]:
import subprocess
import sys
from pathlib import Path

_robust = Path("tests/test_robustness.py")
if not _robust.is_file():
    print(
        f"WARNING: Expected robustness tests at {_robust.resolve()} but file is missing. "
        "Skipping pytest preflight."
    )
else:
    print("Running robustness pytest (excluding GPU model tests)...")
    r = subprocess.run(
        [
            sys.executable,
            "-m",
            "pytest",
            "tests/test_robustness.py",
            "-k",
            "not model_and_tokenizer",
            "-v",
            "--tb=short",
        ],
        cwd=str(Path.cwd()),
    )
    if r.returncode != 0:
        raise RuntimeError(
            "tests/test_robustness.py failed; fix tests before loading the large model."
        )
    print("Robustness tests passed.")


In [ ]:
from src.data.preprocess import load_eval_test_samples

test_samples = load_eval_test_samples(RAW_DATA_PATH)
print(f"Test split: {len(test_samples)} samples")
if test_samples:
    s0 = test_samples[0]
    print("Example keys:", sorted(s0.keys()))
    print("task:", s0.get("task"))
    print("answer (prefix):", str(s0.get("answer", ""))[:120])

Test split: 31 samples
Example keys: ['answer', 'context', 'expression', 'financial_data', 'format', 'id', 'instruction', 'question', 'reasoning', 'task', 'unit', 'variables']
task: numerical_reasoning
answer (prefix): 15.89%


In [ ]:
pip install -U bitsandbytes>=0.46.1

In [ ]:
import torch
from peft import PeftModel
from src.model.load_model import load_model_and_tokenizer, DEFAULT_BNB_CONFIG

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# try:
#     import flash_attn  # noqa: F401
#     _attn = "flash_attention_2"
# except ImportError:
#     _attn = "eager"
_attn = "eager"
eval_base, eval_tokenizer = load_model_and_tokenizer(
    MODEL_ID, DEFAULT_BNB_CONFIG, attn_implementation=_attn
)
eval_model = PeftModel.from_pretrained(eval_base, ADAPTER_DIR)
eval_model.eval()

if torch.cuda.is_available():
    print(f"Model loaded. VRAM: {torch.cuda.memory_allocated() / 1e9:.1f} GB")
else:
    print("Model loaded (CPU — generation will be slow).")

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/579 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded. VRAM: 11.2 GB


In [ ]:
import pandas as pd
from src.eval.evaluate import evaluate_model

metrics_by_mode = evaluate_model(
    model=eval_model,
    tokenizer=eval_tokenizer,
    test_dataset=test_samples,
    output_csv=OUTPUT_CSV,
    max_new_tokens=MAX_NEW_TOKENS,
    max_samples=MAX_SAMPLES,
    run_baseline=True,
    inference_modes=["greedy", "cot", "self_consistency"],
)

print("\nEvaluation Results (per inference mode):")
print("-" * 40)
for mode, m in metrics_by_mode.items():
    print(f"\n--- {mode} ---")
    for k, v in m.items():
        if isinstance(v, float):
            print(f"  {k:<36}: {v:.4f}")
        else:
            print(f"  {k:<36}: {v}")

_delta_rows = []
for mode, m in metrics_by_mode.items():
    row = {"inference_mode": mode}
    for k in (
        "delta_exact_match",
        "delta_f1",
        "delta_parsability_rate",
        "delta_grounding_rate",
    ):
        if k in m:
            row[k] = m[k]
    if len(row) > 1:
        _delta_rows.append(row)
if _delta_rows:
    print("\nDelta (fine-tuned minus baseline):")
    display(pd.DataFrame(_delta_rows))

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



Evaluation Results:
----------------------------------------
  exact_match                 : 0.3226
  f1                          : 0.3710
  parsability_rate            : 1.0000
  grounding_rate              : 0.2581
  n_samples                   : 31.0000
  numerical_reasoning_em      : 0.2000
  numerical_reasoning_f1      : 0.2000
  financial_qa_em             : 0.3810
  financial_qa_f1             : 0.4524


In [ ]:
import pandas as pd

results_df = pd.read_csv(OUTPUT_CSV)
# Avoid pandas inferring empty columns as numeric NaN
for col in ("question", "ground_truth", "prediction"):
    if col in results_df.columns:
        results_df[col] = results_df[col].fillna("").astype(str)

print(f"Total evaluated: {len(results_df)} rows (samples × inference modes)\n")

agg_cols = [c for c in ("exact_match", "f1", "parsable", "grounding_rate") if c in results_df.columns]
if "inference_mode" in results_df.columns and agg_cols:
    print("Mean metrics by task and inference_mode (pivot on exact_match):")
    _g = results_df.groupby(["task", "inference_mode"], observed=False)[agg_cols].mean()
    display(_g["exact_match"].unstack("inference_mode").round(4))
    print("\nFull grouped means:")
    display(_g.round(4))
else:
    print("Per-task breakdown:")
    print(results_df.groupby("task")[agg_cols].mean().round(4))

bad = results_df[results_df["exact_match"] == 0]
print("\nLow-scoring samples (EM=0), first 15:")
display(bad[["task", "question", "ground_truth", "prediction", "f1", "grounding_rate"]].head(15))

Total evaluated: 31 samples

Per-task breakdown:
                     exact_match      f1  parsable  grounding_rate
task                                                              
financial_qa               0.381  0.4524       1.0           0.381
numerical_reasoning        0.200  0.2000       1.0           0.000

Low-scoring samples (EM=0), first 15:


,task,question,ground_truth,prediction,f1,grounding_rate
0,numerical_reasoning,What is the EBITDA margin of Berkshire Hathawa...,15.89%,15.88% %,0.0,0.0
1,numerical_reasoning,What is the free cash flow of Berkshire Hathaw...,7558.612035335822,7558.61195667464 $M,0.0,0.0
2,financial_qa,What was JPMorgan Chase's free cash flow in 2021?,$34.7 billion,$7.4 billion,0.5,0.0
4,financial_qa,What was JPMorgan Chase's gross profit margin ...,62.8%,59.5%,0.0,0.0
6,numerical_reasoning,What is the year-over-year revenue growth rate...,19.63%,19.57% %,0.0,0.0
8,numerical_reasoning,What is the free cash flow of Berkshire Hathaw...,6440.165008787393,6440.164967467746 $M,0.0,0.0
9,financial_qa,How much did Tesla's revenue grow from 2022 to...,17.9%,-0.9%,0.0,0.0
10,numerical_reasoning,What percentage of revenue did Visa spend on R...,7.66%,7.65% %,0.0,0.0
11,financial_qa,What was Apple's gross profit margin in 2022?,66.8%,59.0%,0.0,0.0
12,numerical_reasoning,What is the EBITDA margin of Johnson & Johnson...,11.76%,11.75% %,0.0,0.0
